# Week 62, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w062d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
a = [1]
b = a
a += [2]
a = a + [3]
print(a, b)

**Predict before you run:** what will this print?



In [ ]:
x = [[1], 2]
y = x[:]
print(x == y, x is y, x[0] is y[0])
y[0].append(9)
y[1] = 5
print(x)

**Predict before you run:** what will this print?



In [ ]:
import copy
inner = [0]
outer = [inner, inner, (1, inner)]
c = copy.deepcopy(outer)
print(c[0] is c[1], c[2][1] is c[0], c[0] is inner)

**Predict before you run:** what will this print?



In [ ]:
import gc
gc.disable()
log = []
class P:
    def __init__(self, n):
        self.n = n
    def __del__(self):
        log.append(self.n)
a, b, c = P(1), P(2), P(3)
a.other = a
b.next = c
del a, b
print(log)
gc.collect()
print(log)
gc.enable()

**Predict before you run:** what will this print?



In [ ]:
import weakref
class Item:
    pass
i = Item()
cache = weakref.WeakValueDictionary({"k": i})
r = weakref.ref(i)
print(len(cache), r() is i)
i = Item()
print(len(cache), r())

### Exercise: Pausing the collector

Write a context manager `gc_paused()` (with `@contextmanager`, Week 61) that switches off automatic cycle collection for its block and afterwards **restores the previous state**: if collection was already off before, it stays off. It must work when nested and when the block raises.

In [ ]:
import gc
from contextlib import contextmanager

@contextmanager
def gc_paused():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_paused():\n    \"a context manager around gc\"\n    import gc\n    gc.enable()\n    try:\n        with gc_paused():\n            assert not gc.isenabled()\n            with gc_paused():\n                assert not gc.isenabled()\n            assert not gc.isenabled()\n        assert gc.isenabled()\n        try:\n            with gc_paused():\n                raise ValueError\n        except ValueError:\n            pass\n        assert gc.isenabled()\n        gc.disable()\n        with gc_paused():\n            pass\n        assert not gc.isenabled()\n    finally:\n        gc.enable()")

### Exercise: A memo that forgets

Write a decorator `weak_memo` (Week 60) for functions of **one** argument, an object that can be weakly referenced: it remembers each object's result in a `weakref.WeakKeyDictionary`, so the function runs once per object and the cache never keeps an object alive. Keep the function's name and docstring with `functools.wraps`, and give the wrapper a `cache_size()` function returning the number of cached results.

In [ ]:
import functools
import weakref

def weak_memo(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_weak_memo():\n    \"a decorator with a weak cache\"\n    calls = []\n    @weak_memo\n    def area(shape):\n        \"Area of a shape.\"\n        calls.append(shape.name)\n        return shape.w * shape.h\n    class Shape:\n        def __init__(self, name, w, h):\n            self.name, self.w, self.h = name, w, h\n    a, b = Shape(\"a\", 2, 3), Shape(\"b\", 4, 5)\n    assert area(a) == 6 and area(a) == 6 and area(b) == 20 and calls == [\"a\", \"b\"]\n    assert area.cache_size() == 2 and area.__name__ == \"area\" and area.__doc__ == \"Area of a shape.\"\n    del a\n    assert area.cache_size() == 1")

### Exercise: Forge: an identity map

Write `Repository` to the specification above.

In [ ]:
import copy
import weakref
from contextlib import contextmanager

class Repository:
    def __init__(self, loader):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_identity_map():\n    \"one live object per key\"\n    class Record:\n        def __init__(self, key):\n            self.key, self.data = key, {\"tags\": []}\n    repo = Repository(Record)\n    a = repo.get(\"a\")\n    assert repo.get(\"a\") is a and repo.loads == 1 and repo.live() == [\"a\"]\n    b = repo.get(\"b\")\n    assert repo.live() == [\"a\", \"b\"] and repo.meta(b) == {\"key\": \"b\", \"load\": 2}\n    del b\n    assert repo.live() == [\"a\"] and repo.evicted == [\"b\"]\n    b2 = repo.get(\"b\")\n    assert repo.loads == 3 and repo.meta(b2)[\"load\"] == 3 and repo.evicted == [\"b\"]\n    try:\n        repo.meta(Record(\"x\"))\n        assert False\n    except KeyError:\n        pass\n    m = repo.meta(a)\n    m[\"key\"] = \"changed\"\n    assert repo.meta(a)[\"key\"] == \"a\"\n    bad = Repository(lambda key: [key])\n    try:\n        bad.get(\"x\")\n        assert False\n    except TypeError:\n        pass\n\ndef test_pins():\n    \"pinning keeps objects alive\"\n    class Record:\n        def __init__(self, key):\n            self.key, self.data = key, {\"tags\": []}\n    repo = Repository(Record)\n    assert repo.pin(\"p\") is repo.pin(\"p\")\n    assert repo.live() == [\"p\"] and repo.pinned() == [\"p\"] and repo.loads == 1\n    repo.unpin(\"p\")\n    assert repo.live() == [\"p\"] and repo.pinned() == [\"p\"]\n    repo.unpin(\"p\")\n    assert repo.live() == [] and repo.pinned() == [] and repo.evicted == [\"p\"]\n    try:\n        repo.unpin(\"p\")\n        assert False\n    except ValueError:\n        pass\n    held = repo.get(\"q\")\n    repo.pin(\"q\")\n    del held\n    assert repo.live() == [\"q\"] and repo.loads == 2\n\ndef test_edit():\n    \"copy, edit, commit or abandon\"\n    class Record:\n        def __init__(self, key):\n            self.key, self.data = key, {\"tags\": []}\n    repo = Repository(Record)\n    a = repo.get(\"a\")\n    with repo.edit(\"a\") as draft:\n        assert draft is not a and draft.data == a.data and draft.data is not a.data\n        draft.data[\"tags\"].append(\"new\")\n        assert a.data[\"tags\"] == []\n    assert a.data[\"tags\"] == [\"new\"] and repo.get(\"a\") is a and repo.version(\"a\") == 1\n    try:\n        with repo.edit(\"a\") as draft:\n            draft.data[\"tags\"].clear()\n            raise ValueError(\"abandon\")\n        assert False\n    except ValueError:\n        pass\n    assert a.data[\"tags\"] == [\"new\"] and repo.version(\"a\") == 1\n    with repo.edit(\"a\") as draft:\n        draft.extra = 5\n    assert a.extra == 5 and repo.version(\"a\") == 2 and repo.version(\"never\") == 0\n    with repo.edit(\"z\") as draft:\n        draft.data[\"tags\"].append(\"z\")\n        assert repo.live() == [\"a\", \"z\"]\n    assert repo.version(\"z\") == 1")

> **Who owns what.** Every part of today's repository is an answer to "who keeps this alive?". The weak dictionary means the repository never decides an object's lifetime; the code using the object does. Pins are an explicit, counted decision to keep something. Metadata rides along in a weak side table, so it can't become the reason an object survives. And the edit works on a deep copy, so a half-finished change can't leak into the shared object: sharing is the point of an identity map, so changes to the shared object have to be all or nothing. When you design anything that holds references to other people's objects, ask the same question for each reference.

### Exercise: Forge bonus: copy-on-write

Copying a big list just in case is wasteful when most copies are never changed. **Copy-on-write** shares the data until someone writes. Write `CowList(items=())`:

- The items live in a shared **buffer** object that records how many `CowList`s own it. `CowList(items)` copies `items` into a new buffer (so the caller's list isn't shared).
- `copy()` returns a new `CowList` sharing the **same** buffer, at no cost: no items are copied.
- Reads (`len`, indexing, iteration, `==` with a list or another `CowList`, `repr` as `CowList([1, 2])`) use the buffer directly.
- Writes (`append`, `extend`, item assignment, `pop(i=-1)`) first make sure this list is the buffer's **only** owner: if not, it copies the items into a new buffer of its own and adds one to its `copied` counter (which starts at 0). A sole owner writes in place.
- `owners()` is the number of owners of this list's buffer, and `shares_with(other)` is True if two lists share a buffer.
- When a `CowList` is garbage collected, its buffer must lose an owner, so the remaining owner can write without copying. Use `weakref.finalize` (and `detach()` the old finaliser when a list moves to a new buffer).

In [ ]:
import weakref

class CowList:
    def __init__(self, items=()):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_copy_on_write():\n    \"share until written\"\n    a = CowList([1, 2])\n    b = a.copy()\n    assert a.shares_with(b) and a.owners() == 2 and b == [1, 2] and a == b\n    a.append(3)\n    assert a == [1, 2, 3] and b == [1, 2] and a.copied == 1 and not a.shares_with(b)\n    assert a.owners() == 1 and b.owners() == 1\n    b.append(4)\n    assert b.copied == 0 and b == [1, 2, 4]\n    c = b.copy()\n    d = c.copy()\n    assert b.owners() == 3 and d.shares_with(b)\n    del c\n    assert b.owners() == 2\n    del d\n    b[0] = 9\n    assert b.copied == 0 and b[0] == 9 and len(b) == 3 and list(b) == [9, 2, 4]\n    e = b.copy()\n    assert e.pop() == 4 and e.copied == 1 and b == [9, 2, 4] and e == [9, 2]\n    assert repr(e) == \"CowList([9, 2])\"\n    e.extend([7, 8])\n    assert e.copied == 1 and e == [9, 2, 7, 8]\n\ndef test_lifetimes():\n    \"owners follow object lifetimes\"\n    a = CowList([\"x\"])\n    copies = [a.copy() for _ in range(5)]\n    assert a.owners() == 6\n    copies.clear()\n    assert a.owners() == 1\n    b = a.copy()\n    b.append(\"y\")\n    assert a.owners() == 1 and b.owners() == 1\n    del b\n    assert a.owners() == 1\n    a.append(\"z\")\n    assert a.copied == 0\n    source = [\"s\"]\n    c = CowList(source)\n    source.append(\"t\")\n    assert c == [\"s\"] and CowList([1]) != [2] and CowList([1]) == CowList([1])\n    import gc\n    f = a.copy()\n    f.me = f\n    del f\n    gc.collect()\n    assert a.owners() == 1")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [names and references](w062d1.html), [identity and mutability](w062d2.html), [garbage collection](w062d3.html), [shallow and deep copies](w062d4.html) or [weak references](w062d5.html).

## Mixed review

## Forge: an identity map

An **identity map** makes sure each record is represented by exactly **one** live object, so that a change made through one part of a program is seen by every other part holding that record. Write `Repository(loader)`:

- `get(key)`: return the live object for `key` if there is one; otherwise call `loader(key)`, remember the object **weakly**, and return it. `loads` counts calls to the loader. (Loaders that return objects which can't be weakly referenced make `get` raise `TypeError`, which `weakref` does for you.)
- `live()`: the sorted keys whose objects are alive. `evicted`: a list of keys, in order, appended to **when each loaded object dies** (use `weakref.finalize`).
- `meta(obj)`: a **copy** of the metadata dict `{"key": key, "load": n}` for an object this repository loaded, where `n` is the value of `loads` just after loading it; `KeyError` for other objects. Store metadata in a `WeakKeyDictionary`.
- `pin(key)` returns the object and keeps it alive (strongly) until a matching number of `unpin(key)` calls; `unpin` of a key that isn't pinned raises `ValueError`; `pinned()` lists pinned keys, sorted.
- `edit(key)`: a context manager yielding a **deep copy** of the object (loading it if needed, and keeping it alive during the edit). If the block finishes normally, the draft's attributes **replace** the live object's attributes (so everyone holding the object sees the change) and `version(key)` goes up by one; if the block raises, the live object is untouched and the exception propagates. `version(key)` is 0 for keys never edited.

## Forge bonus: copy-on-write

## Weekly quiz

## Week 62 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can explain names, namespaces, call by sharing and what += does, and draw a reference graph. ([Day 1](w062d1.html))
- I can choose between is and ==, use sentinels, and write immutable value classes with consistent hashing. ([Day 2](w062d2.html))
- I can explain reference counting and the cycle collector, use the gc module, and find leaks. ([Day 3](w062d3.html))
- I can predict what shallow and deep copies share, write both by hand with a memo, and control copying with __copy__ and __deepcopy__. ([Day 4](w062d4.html))
- I can use weak references, the weak containers, WeakMethod and weakref.finalize. ([Day 5](w062d5.html))
- I built an identity map and a copy-on-write list. (Today)

Tomorrow is a **rest day**. Next week: **Attribute magic**: descriptors, how properties work, `__getattr__`, metaclasses and class decorators.